## <font color='blue'>Generative AI and LLMs for Natural Language Processing</font>
## <font color='blue'>Creating Intelligent Applications with LangChain and LLMs</font>

## Installing and Loading Packages

In [ ]:
# To upgrade a package, run the command below in the terminal or command prompt:
# pip install -U package_name

# To install an exact package version, run the command below in the terminal or command prompt:
# !pip install package_name==desired_version

# After installing or upgrading the package, restart the Jupyter notebook.

# Install the watermark package.
# This package is used to record the versions of other packages used in this Jupyter notebook.
!pip install -q -U watermark

The package installation process and dependencies are described in the README.txt file

In [ ]:
# Langchain
#!pip install -q langchain

In [ ]:
# Langchain to connect to OpenAI's LLM
#!pip install -q langchain-openai

In [ ]:
# VectorDB
#!pip install -q chromadb

In [ ]:
# Dependency for Langchain
#!pip install -q sqlalchemy

In [ ]:
# Define o USER_AGENT
import os
os.environ["USER_AGENT"] = "Estudo de Caso 2"

In [ ]:
# Imports
import os
import textwrap
import chromadb
import langchain
import sqlalchemy
import langchain_openai
from langchain_openai import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SimpleSequentialChain
from langchain.chains import SequentialChain
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.document_loaders import WebBaseLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
import warnings
warnings.filterwarnings('ignore')

## Defining the working LLM

In [ ]:
# Insert your OpenAI API key here
os.environ['OPENAI_API_KEY'] = "your-api-key-here"

In [ ]:
# Define the LLM to be used in the notebook
# Create an instance of a Large Language Model (LLM), specifically one provided by OpenAI
dsa_llm = OpenAI(temperature = 0.9)

Temperature is a hyperparameter that influences the randomness of the responses generated by the model. A higher temperature value (typically between 0 and 1) encourages more creative and varied responses. On the other hand, a lower temperature tends to cause the model to produce more deterministic and possibly more predictable responses.

In [ ]:
# Send the prompt to the LLM and capture the response
nome = dsa_llm.invoke("I want to open a Japanese food restaurant. Suggest a fancy name for it.")

In this context, the string "I want to open a Japanese food restaurant. Suggest a fancy name for it." serves as the prompt or input to the language model. It describes the task the user wants the model to perform: the creative generation of a name for a new Japanese food restaurant. The model will use its natural language training and prior knowledge to generate a response that fulfills this request.

In [ ]:
print(nome)

## Using Prompt Templates

Prompt templates in the LangChain context refer to structured ways of formatting input for large language models (LLMs) to improve their performance and adherence to desired behaviors.

A prompt template defines a template string with placeholder variables that can be filled dynamically. This allows you to build prompts consistently and programmatically instead of hardcoding complete prompts.

Prompt templates in LangChain provide a structured and extensible way to interface with LLMs, making it easier to explore and optimize prompt strategies to improve language model performance on specific tasks or domains.

In [ ]:
# Define the prompt template
dsa_prompt_template_name = PromptTemplate(
    input_variables = ['cuisine'],
    template = "I want to open a {cuisine} food restaurant. Suggest a fancy name for it.")

The code line above defines a PromptTemplate, a structure that allows the creation of dynamic prompts to be used with Large Language Models (LLMs). This approach is particularly useful when you want to generate custom prompts based on specific variables or when you intend to reuse a prompt format with different datasets.

**input_variables = ['cuisine']**: Defines a list of variables that can be used to fill the template. In this case, there is a single variable named 'cuisine'. This variable acts as a placeholder that will be replaced by a specific value when the template is used.

In [ ]:
# Uses the previously defined template to generate a specific prompt.
# inserting the value "Italiana" in place of the cuisine variable
p = dsa_prompt_template_name.format(cuisine = "Italiana")

In [ ]:
print(p)

## LLMChain Operation Sequences

Chains in LangChain are sequences of operations that can process inputs and generate outputs by combining multiple components, including large language models (LLMs), other chains, and specialized tools or utilities.

An LLMChain is a type of chain that allows you to interact with a large language model (LLM) in a structured way. It provides a simple interface to pass inputs to the LLM and retrieve its outputs.

LLMChain serves as a building block for many other constructs in LangChain, such as agents, tools, and more advanced chain types. By encapsulating LLM interaction logic in a reusable and extensible component, LLMChain simplifies the process of building applications that leverage large language models.

In [ ]:
# Cria a chain
chain = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name)

The line of code above creates an instance of LLMChain, a class designed to chain or sequence operations using an LLM. This instance is configured to use a specific language model and a predefined prompt template. 

In [ ]:
# Invoke the chain by passing a parameter to the prompt
chain.invoke("Mexicana")

In [ ]:
# Cria a chain e ativa o verbose
chain = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name, verbose = True)

In [ ]:
# Invoke the chain by passing a parameter to the prompt
chain.invoke("Mexicana")

## Simple Sequential Chain

A SimpleSequentialChain in LangChain is a type of chain that executes a sequence of components (for example, LLMs, tools, other chains) in a predefined order. It is one of the most basic and commonly used chain types in LangChain.

An example use case for SimpleSequentialChain could be a question-answering system where:

- The first component is an LLM that analyzes the input question.
- The second component is a tool that retrieves relevant documents from a database.
- The third component is another LLM that generates a response based on the question and the retrieved documents.

By chaining these components into a SimpleSequentialChain, you can create a more complex and capable system while maintaining a modular and extensible architecture.

Although SimpleSequentialChain is useful for linear workflows, LangChain also provides other chain types like ConditionalChain and SequentialChain for more complex control flows and branching logic.

In [ ]:
# Define o LLM com temperatura menor
dsa_llm = OpenAI(temperature = 0.6)

In [ ]:
# Create the prompt template
dsa_prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a {cuisine} food restaurant. Suggest a fancy name for it.")

In [ ]:
# Cria a chain
dsa_chain_1 = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name)

In [ ]:
# Create another prompt template
dsa_prompt_template_items = PromptTemplate(
    input_variables = ['restaurant_name'],
    template = """Sugira alguns itens do menu para {restaurant_name}""")

In [ ]:
# Cria a chain
dsa_chain_2 = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_items)

In [ ]:
# Concatena as duas chains
dsa_chain_final = SimpleSequentialChain(chains = [dsa_chain_1, dsa_chain_2])

In [ ]:
# Invoca a chain
dsa_chain_final.invoke("Indiana")

## Sequential Chain

SequentialChain is a more advanced version of SimpleSequentialChain. While SimpleSequentialChain executes a fixed sequence of components, SequentialChain allows dynamic and conditional execution of components based on the outputs of previous components.

An example use case for SequentialChain could be a conversational agent that:

- Uses an LLM to understand user input and determine the appropriate action.
- Conditionally executes different components (for example, database search, API call, calculation) based on the LLM output.
- Optionally asks the user for additional information if needed.
- Generates a final response using another LLM, based on the outputs of the previous components.

By leveraging SequentialChain, you can build more intelligent and adaptable applications that can dynamically adjust their behavior based on intermediate results and states.

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [ ]:
# Define the LLM with a lower temperature
dsa_llm = OpenAI(temperature = 0.7)

In [ ]:
# Creating the first chain

# Define the prompt template
dsa_prompt_template_name = PromptTemplate(
    input_variables = ['cuisine'],
    template = "I want to open a {cuisine} food restaurant. Suggest a fancy name for it.")

# Define a chain with an output parameter
dsa_chain_1 = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name, output_key = "restaurant_name")

In [ ]:
# Creating the second chain

# Define the prompt template
dsa_prompt_template_items = PromptTemplate(
    input_variables = ['restaurant_name'],
    template = "Sugira alguns itens do menu para {restaurant_name}."
)

# Define a chain with an output parameter
dsa_chain_2 = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_items, output_key = "menu_items")

In [ ]:
# Create the sequence of chains
dsa_chain = SequentialChain(chains = [dsa_chain_1, dsa_chain_2],
                            input_variables = ['cuisine'],
                            output_variables = ['restaurant_name', "menu_items"])

In [ ]:
dsa_chain.invoke({"cuisine": "Italiana"})

In [ ]:
# Method invocation and response capture
response = dsa_chain.invoke({"cuisine": "Italiana"})

# Preparing the formatted output
saida_formatada = f"Cuisine: {response['cuisine']}\nRestaurant Name: {response['restaurant_name'].strip()}\nMenu Items:"

# Adding each menu item to the formatted string
menu_items = response['menu_items'].strip().split('\n')
for item in menu_items:
    saida_formatada += f"\n{item}"

# Displaying the formatted output
print(saida_formatada)

## Creating Memory for the LLM

In LangChain, "Memory" refers to components that allow chains, agents, and other constructs to store and retrieve information from previous inputs, outputs, and intermediate states. This enables them to maintain context and make use of relevant information from conversation history or previous computations.

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [ ]:
# Define a chain
dsa_chain = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name)

In [ ]:
# Invoke a chain
nome = dsa_chain.invoke("Mexicana")
print(nome)

In [ ]:
# Invoke a chain
nome = dsa_chain.invoke("Argentina")
print(nome)

In [ ]:
dsa_chain.memory

In [ ]:
type(dsa_chain.memory)

In [ ]:
# Creating the memory object
memory = ConversationBufferMemory()

In [ ]:
dsa_chain = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name, memory = memory)

In [ ]:
nome = dsa_chain.run("Mexicana")
print(nome)

In [ ]:
nome = dsa_chain.run("Argentina")
print(nome)

In [ ]:
print(dsa_chain.memory.buffer)

## Conversation Chain

A ConversationChain is a specialized type of chain designed to handle multi-turn conversations or dialogues with an LLM.

ConversationChain is particularly useful for building conversational agents, chatbots, or any application that requires maintaining context across multiple interaction turns with a user. By abstracting the complexities of conversation history management and prompt formatting, ConversationChain simplifies the process of building multi-turn dialogue systems with LLMs.

In [ ]:
# Create the conversation object
conv = ConversationChain(llm = OpenAI(temperature = 0.7))

In [ ]:
print(conv.prompt.template)

In [ ]:
conv.invoke("Which country has won the FIFA World Cup the most times?")

In [ ]:
conv.invoke("What is 30 + 12?")

In [ ]:
conv.invoke("Who is the all-time top goalscorer in FIFA World Cup history?")

In [ ]:
print(conv.memory.buffer)

## Conversation Buffer Window Memory

ConversationBufferWindowMemory is a type of LangChain memory component designed specifically for use with conversation chains (ConversationChain). It provides a way to store and retrieve conversation history while limiting the amount of context retained based on a specified window size.

The main advantage of ConversationBufferWindowMemory is its ability to limit the amount of context provided to the LLM, which can be important for performance and avoiding overloading the model with irrelevant information. By adjusting the window size, you can control the trade-off between providing enough context and avoiding excessive computational overhead.

This type of memory is particularly useful for creating conversational agents, chatbots, or any application that requires maintaining a continuous relevant window of recent conversation history for context.

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [ ]:
# Define the memory window
memory = ConversationBufferWindowMemory(k = 1)

In [ ]:
# Create the conversation chain
conv = ConversationChain(llm = OpenAI(temperature = 0.7), memory = memory)

In [ ]:
# Invoke the LLM
conv.run("Quem venceu a primeira Copa do Mundo de Futebol?")

In [ ]:
# Invoke the LLM
conv.invoke("What is 10 + 19?")

In [ ]:
# Invoke the LLM
conv.invoke("Who was the captain of the team that won the first FIFA World Cup?")

In [ ]:
print(conv.memory.buffer)

## LangChain and VectorDB with Web Scraped Data

ChromaDB is a vector database library that integrates with LangChain. It provides functionality to store, retrieve, and search large amounts of text data efficiently using vector embeddings and semantic similarity.

https://www.trychroma.com/

https://pypi.org/project/chromadb/

In [ ]:
# Web data extraction
dsa_dados = WebBaseLoader(
    "https://blog.dsacademy.com.br/como-rag-retrieval-augmented-generation-funciona-para-personalizar-os-llms/"
)

Note: Always check a website's robots.txt before scraping data. Do not scrape if it is not allowed!

In [ ]:
# Load the documents
documentos = dsa_dados.load()

In [ ]:
len(documentos)

In [ ]:
# Extract the first document (in this case there is only one document)
document = documentos[0]

In [ ]:
# Dictionary keys
document.__dict__.keys()

In [ ]:
# Visualize the first 100 characters
document.page_content[:100]

In [ ]:
# Metadata
document.metadata

In [ ]:
# Create the vector store (adjusted for the latest LangChain version)
embedding = OpenAIEmbeddings() 
index_creator = VectorstoreIndexCreator(embedding = embedding)
index = index_creator.from_loaders([dsa_dados])
#index = VectorstoreIndexCreator().from_loaders([dsa_dados])

The line of code above involves creating an index for a vector store, commonly used in search and retrieval tasks based on content similarity. This operation is essential in systems that employ AI and machine learning techniques to organize and retrieve data efficiently. 

While a "Vector Store" is a more generic concept related to vector storage, a "Vector Database" is a specialized database system designed to handle high-dimensional vectors and optimized for similarity queries and other vector-related operations. We will use ChromaDB to create a vector database.

In [ ]:
# Function to print formatted response
def dsa_print_response(response: str):
    print("\n".join(textwrap.wrap(response, width = 100)))

In [ ]:
# Define a query
query = """
You are a Senior AI Engineer.
Explain how RAG works in building intelligent applications.
"""

In [ ]:
# Print the response when querying the index (adjusted for the latest LangChain version)
response = index.query(query, llm = OpenAI(temperature = 0.7))
dsa_print_response(response)

> Working with Vector DB.

In [ ]:
# Create a template
dsa_template = """You are a Senior AI Engineer.

{context}

Answer considering the most modern techniques you know.

Peegunta: {question}
Answer:"""

In [ ]:
# Create the prompt
prompt = PromptTemplate(template = dsa_template, input_variables = ["context", "question"])

In [ ]:
# Print the prompt to visualize the format
print(
    prompt.format(
        context = "AI application for customer service systems.",
        question = "How to create a web application with an LLM?",
    )
)

In [ ]:
# Create embeddings object
embeddings = OpenAIEmbeddings()

The line of code above refers to initializing an instance of the OpenAIEmbeddings class, which is an interface for generating embeddings (vector representations) using models provided by OpenAI, such as GPT language models. Embeddings are transformations of raw data, such as text, into fixed-size vectors that capture semantic and contextual aspects of the original content so they can be processed by machine learning algorithms.

In [ ]:
# Create the VectorDB by converting text documents into numerical representations (embeddings)
dsa_db = Chroma.from_documents(documentos, embeddings)

The line of code above is for creating a vector database using Chroma. This operation involves preparing a data structure optimized for search and analysis based on documents and their embeddings. 

In [ ]:
type(dsa_db)

In [ ]:
# Arguments for the chain
chain_type_kwargs = {"prompt": prompt}

In [ ]:
# Chain of RetrievalQA
chain = RetrievalQA.from_chain_type(llm = ChatOpenAI(temperature = 0),
                                    chain_type = "stuff",
                                    retriever = dsa_db.as_retriever(search_kwargs = {"k": 1}),
                                    chain_type_kwargs = chain_type_kwargs)

The line of code above is creating an instance called chain, using the RetrievalQA class to configure a process chain focused on performing Question Answering (QA) tasks based on information retrieval. The from_chain_type method is used to specify the type of process chain and configure its main components, such as the language model and retrieval mechanism. 

In [ ]:
# Consult
query = "Explain what RAG is in 5 sentences"

In [ ]:
# Response
response = chain.invoke(query)

In [ ]:
response

## Creating a Sales Specialist Chatbot with LangChain and LLM

In [ ]:
# Defining the LLM
dsa_gpt = ChatOpenAI(temperature = 0)

In [ ]:
# Template
template = """This is a conversation between a customer and a sports car sales specialist. 
                You are the car specialist, you know sports car models well and should always answer 
                with the greatest accuracy possible.

Current conversation:
{history}
Human: {input}
CarSpecialist:"""

In [ ]:
# Create the prompt template
dsa_prompt = PromptTemplate(input_variables = ["history", "input"], template = template)

In [ ]:
# Create the conversation chain
conversation = ConversationChain(prompt = dsa_prompt,
                                 llm = dsa_gpt,
                                 verbose = False,
                                 memory = ConversationBufferMemory(ai_prefix = "CarSpecialist"))

In [ ]:
# Conversation loop limited to 5 interactions (increase the number of interactions or remove the if block)

# Initialize the counter
contador = 0  

# Loop
while True:
    
    prompt = input(prompt = "Customer: ")
    print()
    resultado = conversation(prompt)
    dsa_print_response("Specialist: " + resultado["response"])
    print()
    
    contador += 1  
    
    if contador >= 5:  
        print('\nThank you for using the AI-based customer service system!')
        break  